# LSTM Inertia-Surrogate Path (Notebook-First) — 11.03.2026

This notebook builds a stronger, fair LSTM evaluation path to answer:

1. Does adding **surrogate inertia-state features** improve global pooled forecasting?
2. Does EHR (`setB`) improve relative to non-EHR (`setA`) under:
   - base dynamic features, and
   - dynamic + surrogate inertia features?

## Why this notebook

- `04_lstm_experiments_10032026.ipynb` established the baseline LSTM comparisons.
- `05_surrogate_indoor_state_experiments_11032026.ipynb` produced a plausible latent indoor-state proxy.
- This notebook combines both into one fair, reproducible experiment matrix with tables and plots.

## Fairness constraints

- Same matched building cohort (EHR-complete core buildings).
- Same train/val/test temporal splits for all variants.
- Same architecture family and optimizer settings across variants.


> Note: this notebook includes a quick-run cap for sequence counts to stay runnable on CPU/M-series laptops.


In [1]:
# Section 0 — Imports and configuration

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy.stats import wilcoxon

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Input, Model, layers

plt.style.use("seaborn-v0_8")
plt.rcParams["figure.figsize"] = (12, 5)

PROJECT_ROOT = Path.cwd()
FEATURE_DIR = PROJECT_ROOT / "data" / "features"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42]             # expand to [42,52,62,72,82] for more robust estimate
SURROGATE_PRESET = "heavy_slow"  # from Notebook 05 summary
RUN_EXPERIMENT = True

@dataclass
class ExpConfig:
    lookback: int = 24
    batch_size: int = 64
    epochs: int = 8
    learning_rate: float = 1e-3
    early_stopping_patience: int = 5
    train_end: str = "2023-09-30 23:00:00"
    val_start: str = "2023-10-01 00:00:00"
    test_start: str = "2024-01-01 00:00:00"

CFG = ExpConfig()

EHR_FEATURES = [
    "ehr_heated_area_m2",
    "ehr_volume_per_heated_area",
    "ehr_compactness_ratio",
    "ehr_building_age_years",
    "ehr_max_floors",
    "ehr_has_heat_recovery",
    "ehr_structural_mass_code",
    "ehr_wall_type_code",
]
EHR_NUMERIC = [
    "ehr_heated_area_m2",
    "ehr_volume_per_heated_area",
    "ehr_compactness_ratio",
    "ehr_building_age_years",
    "ehr_max_floors",
]
EHR_BINARY = ["ehr_has_heat_recovery"]
EHR_CATEGORICAL = ["ehr_structural_mass_code", "ehr_wall_type_code"]

SURROGATE_PRESETS = {
    "light_fast": {"a": 0.16, "b": 0.08, "c": 0.10, "d": 0.03},
    "medium": {"a": 0.11, "b": 0.05, "c": 0.07, "d": 0.02},
    "heavy_slow": {"a": 0.07, "b": 0.03, "c": 0.05, "d": 0.015},
}

assert SURROGATE_PRESET in SURROGATE_PRESETS
print(f"Project root      : {PROJECT_ROOT}")
print(f"RUN_EXPERIMENT    : {RUN_EXPERIMENT}")
print(f"Seeds             : {SEEDS}")
print(f"Surrogate preset  : {SURROGATE_PRESET}")
print(f"Config            : {CFG}")

MAX_SEQ_PER_SPLIT_PER_BUILDING = 6000  # quick-run cap for portability


Project root      : /Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project
RUN_EXPERIMENT    : True
Seeds             : [42]
Surrogate preset  : heavy_slow
Config            : ExpConfig(lookback=24, batch_size=64, epochs=8, learning_rate=0.001, early_stopping_patience=5, train_end='2023-09-30 23:00:00', val_start='2023-10-01 00:00:00', test_start='2024-01-01 00:00:00')


## Section 1 — Cohort Definition

We use a **matched cohort** so that `setA` and `setB` are directly comparable:

- EHR complete (`ehr_complete_any=True`)
- enough test coverage (`test_complete_rows_setB >= 1000`)

This avoids unfair comparisons caused by different building subsets.


In [2]:
# Section 1 — Load metadata and define matched cohort

feature_meta = pd.read_csv(FEATURE_DIR / "feature_metadata.csv")
coverage = pd.read_csv(FEATURE_DIR / "portfolio_coverage.csv")

MATCHED_COHORT = coverage.loc[
    (coverage["ehr_complete_any"].astype(bool)) & (coverage["test_complete_rows_setB"] >= 1000),
    "building",
].tolist()
MATCHED_COHORT = [b for b in MATCHED_COHORT if b in feature_meta["building"].tolist()]

print("Matched cohort (fair setA vs setB):")
print(MATCHED_COHORT)
print(f"n_buildings={len(MATCHED_COHORT)}")


Matched cohort (fair setA vs setB):
['GEO', 'LIB', 'SOC', 'U01', 'U02', 'U02B', 'U03', 'U03B', 'U04', 'U05', 'U06', 'U06A']
n_buildings=12


## Section 2 — Surrogate Inertia Feature Construction

For each building, we derive surrogate latent-state features from clean merged data:

- `sur_t_in_hat_c`
- `sur_t_in_hat_diff1h`
- `sur_comfort_deficit_c`
- `sur_comfort_deficit_roll6h`

These are merged into both `setA` and `setB` feature frames by datetime.


In [3]:
# Section 2 — Helpers: reproducibility, loading, surrogate simulation

def set_all_seeds(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def pick_first_matching_column(columns: List[str], suffixes: List[str]) -> str | None:
    for suffix in suffixes:
        cands = [c for c in columns if c.endswith(suffix)]
        if cands:
            return cands[0]
    return None


def load_clean_for_surrogate(clean_path: Path) -> pd.DataFrame:
    df = pd.read_csv(clean_path, parse_dates=["Time"])
    heat_col = pick_first_matching_column(
        list(df.columns),
        [
            "__weather_driven__energy_delta_mwh",
            "__space_heating__energy_delta_mwh",
            "__total__energy_delta_mwh",
        ],
    )
    if heat_col is None:
        raise ValueError(f"No heat column found in {clean_path.name}")

    if "COP_temp_c" in df.columns:
        temp_col = "COP_temp_c"
    elif "KKP_temp_c" in df.columns:
        temp_col = "KKP_temp_c"
    else:
        raise ValueError(f"No outdoor temp column found in {clean_path.name}")

    if "COP_ssrd_W_per_m2" in df.columns:
        solar_col = "COP_ssrd_W_per_m2"
    elif "KKP_sunshine_duration_min" in df.columns:
        solar_col = "KKP_sunshine_duration_min"
    else:
        solar_col = None

    keep_cols = ["Time", heat_col, temp_col]
    if solar_col is not None:
        keep_cols.append(solar_col)

    out = df[keep_cols].copy().rename(
        columns={
            "Time": "datetime",
            heat_col: "heat_mwh",
            temp_col: "outdoor_temp_c",
            solar_col: "solar_wm2" if solar_col is not None else "solar_wm2",
        }
    )
    if "solar_wm2" not in out.columns:
        out["solar_wm2"] = 0.0

    out["heat_kwh"] = out["heat_mwh"] * 1000.0
    out = out.drop(columns=["heat_mwh"]).sort_values("datetime").set_index("datetime")
    out = out.resample("h").mean()

    out["outdoor_temp_c"] = out["outdoor_temp_c"].interpolate(limit_direction="both")
    out["solar_wm2"] = out["solar_wm2"].fillna(0.0).clip(lower=0.0)
    out["heat_kwh"] = out["heat_kwh"].fillna(0.0).clip(lower=0.0)
    return out


def build_setpoint_series(index: pd.DatetimeIndex) -> pd.Series:
    hours = index.hour
    is_day = (hours >= 6) & (hours < 22)
    vals = np.where(is_day, 21.0, 20.0)
    return pd.Series(vals, index=index, dtype=float)


def simulate_surrogate_state(clean_df: pd.DataFrame, params: Dict[str, float]) -> pd.DataFrame:
    out = clean_df.copy()
    out["t_set_c"] = build_setpoint_series(out.index)
    out["heat_lag1_kwh"] = out["heat_kwh"].shift(1).bfill()

    q_scale = float(np.nanpercentile(out["heat_lag1_kwh"], 95))
    if not np.isfinite(q_scale) or q_scale <= 0:
        q_scale = 1.0

    s_scale = float(np.nanpercentile(out["solar_wm2"], 95))
    if not np.isfinite(s_scale) or s_scale <= 0:
        s_scale = 1.0

    q_norm = (out["heat_lag1_kwh"] / q_scale).values
    s_norm = (out["solar_wm2"] / s_scale).values
    t_out = out["outdoor_temp_c"].values
    t_set = out["t_set_c"].values

    a = float(params["a"])
    b = float(params["b"])
    c = float(params["c"])
    d = float(params["d"])

    t_hat = np.empty(len(out), dtype=float)
    t_hat[0] = t_set[0]
    for i in range(1, len(out)):
        prev = t_hat[i - 1]
        t_hat[i] = (
            prev
            + a * (t_set[i - 1] - prev)
            + b * (t_out[i - 1] - prev)
            + c * q_norm[i - 1]
            + d * s_norm[i - 1]
        )

    t_hat = np.clip(t_hat, 10.0, 30.0)
    out["sur_t_in_hat_c"] = t_hat
    out["sur_t_in_hat_diff1h"] = out["sur_t_in_hat_c"].diff().fillna(0.0)
    out["sur_comfort_deficit_c"] = out["t_set_c"] - out["sur_t_in_hat_c"]
    out["sur_comfort_deficit_roll6h"] = out["sur_comfort_deficit_c"].rolling(6, min_periods=1).mean()

    return out[
        [
            "sur_t_in_hat_c",
            "sur_t_in_hat_diff1h",
            "sur_comfort_deficit_c",
            "sur_comfort_deficit_roll6h",
        ]
    ].copy()


In [4]:
# Section 2b — Load setA/setB feature frames and merge surrogate features

frames_setA: Dict[str, pd.DataFrame] = {}
frames_setB: Dict[str, pd.DataFrame] = {}
surrogate_by_building: Dict[str, pd.DataFrame] = {}

for b in MATCHED_COHORT:
    row = feature_meta.loc[feature_meta["building"] == b].iloc[0]
    dfA = pd.read_csv(PROJECT_ROOT / row["path_setA"], parse_dates=["datetime"]).set_index("datetime").sort_index()
    dfB = pd.read_csv(PROJECT_ROOT / row["path_setB"], parse_dates=["datetime"]).set_index("datetime").sort_index()

    clean_df = load_clean_for_surrogate(CLEAN_DIR / f"{b}_hourly_merged.csv")
    sur = simulate_surrogate_state(clean_df, SURROGATE_PRESETS[SURROGATE_PRESET])

    frames_setA[b] = dfA
    frames_setB[b] = dfB
    surrogate_by_building[b] = sur

print("Loaded feature frames and surrogate features for all matched buildings.")
print(f"Example shape setA[{MATCHED_COHORT[0]}]:", frames_setA[MATCHED_COHORT[0]].shape)
print(f"Example shape surrogate[{MATCHED_COHORT[0]}]:", surrogate_by_building[MATCHED_COHORT[0]].shape)


Loaded feature frames and surrogate features for all matched buildings.
Example shape setA[GEO]: (22884, 23)
Example shape surrogate[GEO]: (22886, 4)


## Section 3 — Model and Sequence Helpers

Two model variants are used:

- `temporal-only` for setA variants
- `temporal + static EHR branch` for setB variants

Variants to run:

- `setA_base`
- `setB_base`
- `setA_sur`
- `setB_sur`


In [5]:
# Section 3 — LSTM + sequence helpers

def build_lstm_temporal_only(n_timesteps: int, n_features: int, lr: float) -> keras.Model:
    temporal_in = Input(shape=(n_timesteps, n_features), name="temporal_input")
    x = layers.LSTM(64, return_sequences=True, name="lstm_1")(temporal_in)
    x = layers.Dropout(0.2, name="dropout_1")(x)
    x = layers.LSTM(32, name="lstm_2")(x)
    x = layers.Dense(16, activation="relu", name="dense_1")(x)
    out = layers.Dense(1, name="output")(x)
    model = Model(inputs=temporal_in, outputs=out, name="lstm_temporal_only")
    # Use standard Keras 3 optimizer API (no legacy namespace)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss="mse", metrics=["mse"])
    return model


def build_lstm_temporal_plus_ehr(n_timesteps: int, n_dynamic: int, n_static_ehr: int, lr: float) -> keras.Model:
    temporal_in = Input(shape=(n_timesteps, n_dynamic), name="temporal_input")
    static_in = Input(shape=(n_static_ehr,), name="ehr_input")

    x = layers.LSTM(64, return_sequences=True, name="lstm_1")(temporal_in)
    x = layers.Dropout(0.2, name="dropout_1")(x)
    x = layers.LSTM(32, name="lstm_2")(x)

    s = layers.Dense(16, activation="relu", name="ehr_dense_1")(static_in)
    s = layers.Dense(8, activation="relu", name="ehr_dense_2")(s)

    merged = layers.Concatenate(name="concat")([x, s])
    h = layers.Dense(16, activation="relu", name="merged_dense_1")(merged)
    out = layers.Dense(1, name="output")(h)

    model = Model(inputs=[temporal_in, static_in], outputs=out, name="lstm_temporal_plus_ehr")
    # Use standard Keras 3 optimizer API (no legacy namespace)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss="mse", metrics=["mse"])
    return model


def build_sequences(df: pd.DataFrame, dynamic_cols: List[str], target_col: str, split_mask: np.ndarray, lookback: int):
    mask = np.asarray(split_mask, dtype=bool)
    X_list, y_list = [], []

    for end_idx in range(lookback, len(df)):
        if not mask[end_idx]:
            continue
        start_idx = end_idx - lookback
        if not mask[start_idx:end_idx].all():
            continue

        window = df.iloc[start_idx:end_idx]
        yv = df.iloc[end_idx][target_col]

        if window[dynamic_cols].isna().any().any() or pd.isna(yv):
            continue

        X_list.append(window[dynamic_cols].values.astype("float32"))
        y_list.append(float(yv))

    if not X_list:
        return np.empty((0, lookback, len(dynamic_cols)), dtype="float32"), np.empty((0,), dtype="float32")

    return np.stack(X_list, axis=0), np.array(y_list, dtype="float32")


def wape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.sum(np.abs(y_true))
    if denom <= 0:
        return np.nan
    return float(100.0 * np.sum(np.abs(y_true - y_pred)) / denom)


def fit_ehr_encoders(frames_setB: Dict[str, pd.DataFrame], buildings: List[str], cfg: ExpConfig):
    train_end = pd.Timestamp(cfg.train_end)
    num_rows = []
    cat_rows = []

    for b in buildings:
        df = frames_setB[b]
        train_mask = df.index <= train_end
        block = df.loc[train_mask, EHR_FEATURES].dropna()
        if block.empty:
            continue
        num_rows.append(block[EHR_NUMERIC + EHR_BINARY].values)
        cat_rows.append(block[EHR_CATEGORICAL].values)

    if not num_rows:
        raise RuntimeError("No complete EHR rows found to fit encoders.")

    X_num = np.vstack(num_rows)
    X_cat = np.vstack(cat_rows)

    num_scaler = StandardScaler()
    num_scaler.fit(X_num)

    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    ohe.fit(X_cat)

    return num_scaler, ohe


def build_static_vector(df_setB: pd.DataFrame, num_scaler: StandardScaler, ohe: OneHotEncoder) -> np.ndarray:
    block = df_setB[EHR_FEATURES].dropna()
    if block.empty:
        raise RuntimeError("No complete EHR row for building static vector.")

    row = block.iloc[[0]]
    x_num = row[EHR_NUMERIC + EHR_BINARY].values
    x_cat = row[EHR_CATEGORICAL].values

    x_num_scaled = num_scaler.transform(x_num)
    x_cat_oh = ohe.transform(x_cat)

    return np.hstack([x_num_scaled, x_cat_oh]).astype("float32")


In [6]:
# Section 4 — Variant runner (global pooled)

def run_global_variant(
    variant: str,
    cfg: ExpConfig,
    buildings: List[str],
    frames_setA: Dict[str, pd.DataFrame],
    frames_setB: Dict[str, pd.DataFrame],
    surrogate_by_building: Dict[str, pd.DataFrame],
    num_scaler_ehr: StandardScaler | None,
    ohe_ehr: OneHotEncoder | None,
):
    use_setB = "setB" in variant
    use_sur = "sur" in variant
    use_ehr = use_setB

    train_end = pd.Timestamp(cfg.train_end)
    val_start = pd.Timestamp(cfg.val_start)
    test_start = pd.Timestamp(cfg.test_start)

    feature_scalers: Dict[str, StandardScaler] = {}
    target_scalers: Dict[str, StandardScaler] = {}

    X_tr_dyn_list, y_tr_list = [], []
    X_va_dyn_list, y_va_list = [], []
    X_te_dyn_list, y_te_list = [], []
    b_te_list = []

    X_tr_ehr_list, X_va_ehr_list, X_te_ehr_list = [], [], []

    n_train_total = 0
    n_val_total = 0
    n_test_total = 0

    for b in buildings:
        base_df = frames_setB[b].copy() if use_setB else frames_setA[b].copy()
        if use_sur:
            base_df = base_df.join(surrogate_by_building[b], how="left")

        dynamic_cols = [c for c in base_df.columns if c != "heat_kwh" and not c.startswith("ehr_")]

        train_mask = base_df.index <= train_end
        val_mask = (base_df.index >= val_start) & (base_df.index < test_start)
        test_mask = base_df.index >= test_start

        f_scaler = StandardScaler()
        t_scaler = StandardScaler()

        f_scaler.fit(base_df.loc[train_mask, dynamic_cols].values)
        t_scaler.fit(base_df.loc[train_mask, ["heat_kwh"]].values)

        feature_scalers[b] = f_scaler
        target_scalers[b] = t_scaler

        df_scaled = base_df.copy()
        df_scaled[dynamic_cols] = f_scaler.transform(base_df[dynamic_cols].values)
        df_scaled["heat_kwh_scaled"] = t_scaler.transform(base_df[["heat_kwh"]].values)

        X_tr, y_tr = build_sequences(df_scaled, dynamic_cols, "heat_kwh_scaled", np.asarray(train_mask, dtype=bool), cfg.lookback)
        X_va, y_va = build_sequences(df_scaled, dynamic_cols, "heat_kwh_scaled", np.asarray(val_mask, dtype=bool), cfg.lookback)
        X_te, y_te = build_sequences(df_scaled, dynamic_cols, "heat_kwh_scaled", np.asarray(test_mask, dtype=bool), cfg.lookback)

        if X_tr.shape[0] == 0 or X_va.shape[0] == 0 or X_te.shape[0] == 0:
            print(f"[WARN] Skipping {b} in {variant} due to empty split sequences")
            continue


        # Optional cap for quick experimentation on slower hardware
        if MAX_SEQ_PER_SPLIT_PER_BUILDING is not None:
            if X_tr.shape[0] > MAX_SEQ_PER_SPLIT_PER_BUILDING:
                idx = np.random.choice(X_tr.shape[0], size=MAX_SEQ_PER_SPLIT_PER_BUILDING, replace=False)
                X_tr, y_tr = X_tr[idx], y_tr[idx]
            if X_va.shape[0] > MAX_SEQ_PER_SPLIT_PER_BUILDING:
                idx = np.random.choice(X_va.shape[0], size=MAX_SEQ_PER_SPLIT_PER_BUILDING, replace=False)
                X_va, y_va = X_va[idx], y_va[idx]
            if X_te.shape[0] > MAX_SEQ_PER_SPLIT_PER_BUILDING:
                idx = np.random.choice(X_te.shape[0], size=MAX_SEQ_PER_SPLIT_PER_BUILDING, replace=False)
                X_te, y_te = X_te[idx], y_te[idx]

        X_tr_dyn_list.append(X_tr)
        y_tr_list.append(y_tr)
        X_va_dyn_list.append(X_va)
        y_va_list.append(y_va)
        X_te_dyn_list.append(X_te)
        y_te_list.append(y_te)
        b_te_list.extend([b] * X_te.shape[0])

        n_train_total += int(X_tr.shape[0])
        n_val_total += int(X_va.shape[0])
        n_test_total += int(X_te.shape[0])

        if use_ehr:
            assert num_scaler_ehr is not None and ohe_ehr is not None
            static_vec = build_static_vector(frames_setB[b], num_scaler_ehr, ohe_ehr)
            X_tr_ehr_list.append(np.repeat(static_vec, repeats=X_tr.shape[0], axis=0))
            X_va_ehr_list.append(np.repeat(static_vec, repeats=X_va.shape[0], axis=0))
            X_te_ehr_list.append(np.repeat(static_vec, repeats=X_te.shape[0], axis=0))

    if not X_tr_dyn_list:
        raise RuntimeError(f"No pooled sequences for variant {variant}")

    X_train_dyn = np.concatenate(X_tr_dyn_list, axis=0)
    y_train = np.concatenate(y_tr_list, axis=0)
    X_val_dyn = np.concatenate(X_va_dyn_list, axis=0)
    y_val = np.concatenate(y_va_list, axis=0)
    X_test_dyn = np.concatenate(X_te_dyn_list, axis=0)
    y_test = np.concatenate(y_te_list, axis=0)

    if use_ehr:
        X_train_ehr = np.concatenate(X_tr_ehr_list, axis=0)
        X_val_ehr = np.concatenate(X_va_ehr_list, axis=0)
        X_test_ehr = np.concatenate(X_te_ehr_list, axis=0)

        model = build_lstm_temporal_plus_ehr(
            n_timesteps=cfg.lookback,
            n_dynamic=X_train_dyn.shape[2],
            n_static_ehr=X_train_ehr.shape[1],
            lr=cfg.learning_rate,
        )

        fit_inputs = [X_train_dyn, X_train_ehr]
        val_inputs = ([X_val_dyn, X_val_ehr], y_val)
        pred_inputs = [X_test_dyn, X_test_ehr]
    else:
        model = build_lstm_temporal_only(
            n_timesteps=cfg.lookback,
            n_features=X_train_dyn.shape[2],
            lr=cfg.learning_rate,
        )

        fit_inputs = X_train_dyn
        val_inputs = (X_val_dyn, y_val)
        pred_inputs = X_test_dyn

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=cfg.early_stopping_patience,
            restore_best_weights=True,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            verbose=0,
        ),
    ]

    model.fit(
        fit_inputs,
        y_train,
        validation_data=val_inputs,
        epochs=cfg.epochs,
        batch_size=cfg.batch_size,
        verbose=0,
        callbacks=callbacks,
    )

    y_pred_scaled = model.predict(pred_inputs, verbose=0).reshape(-1, 1)

    per_building_rows = []
    all_true, all_pred = [], []

    for b in sorted(set(b_te_list)):
        idx = [i for i, bb in enumerate(b_te_list) if bb == b]
        y_true_scaled_b = y_test[idx]
        y_pred_scaled_b = y_pred_scaled[idx]

        scaler_y = target_scalers[b]
        y_true_b = scaler_y.inverse_transform(y_true_scaled_b.reshape(-1, 1)).flatten()
        y_pred_b = scaler_y.inverse_transform(y_pred_scaled_b.reshape(-1, 1)).flatten()

        all_true.append(y_true_b)
        all_pred.append(y_pred_b)

        per_building_rows.append({
            "building": b,
            "variant": variant,
            "rmse": float(np.sqrt(mean_squared_error(y_true_b, y_pred_b))),
            "mae": float(mean_absolute_error(y_true_b, y_pred_b)),
            "r2": float(r2_score(y_true_b, y_pred_b)),
            "wape_pct": wape(y_true_b, y_pred_b),
            "n_test_seq": int(len(y_true_b)),
        })

    y_true_all = np.concatenate(all_true)
    y_pred_all = np.concatenate(all_pred)

    portfolio = {
        "variant": variant,
        "rmse": float(np.sqrt(mean_squared_error(y_true_all, y_pred_all))),
        "mae": float(mean_absolute_error(y_true_all, y_pred_all)),
        "r2": float(r2_score(y_true_all, y_pred_all)),
        "wape_pct": wape(y_true_all, y_pred_all),
        "n_train_seq": n_train_total,
        "n_val_seq": n_val_total,
        "n_test_seq": n_test_total,
    }

    return pd.DataFrame(per_building_rows), portfolio


## Section 5 — Run Experiment Matrix

Variants:

- `setA_base` : no EHR, no surrogate features
- `setB_base` : EHR branch, no surrogate features
- `setA_sur`  : no EHR + surrogate features
- `setB_sur`  : EHR branch + surrogate features

Interpretation target:

- EHR relative gain (pp): `setA_wape - setB_wape`
  - positive => EHR improves
  - negative => EHR worsens


In [7]:
# Section 5 — Execute matrix

variant_list = ["setA_base", "setB_base", "setA_sur", "setB_sur"]

if not RUN_EXPERIMENT:
    print("RUN_EXPERIMENT=False. Set it True to execute matrix.")
else:
    all_detail = []
    all_portfolio = []

    for seed in SEEDS:
        print(f"\n=== Running seed {seed} ===")
        set_all_seeds(seed)

        num_scaler_ehr, ohe_ehr = fit_ehr_encoders(frames_setB, MATCHED_COHORT, CFG)

        for variant in variant_list:
            print(f"  -> variant: {variant}")
            detail_df, portfolio = run_global_variant(
                variant=variant,
                cfg=CFG,
                buildings=MATCHED_COHORT,
                frames_setA=frames_setA,
                frames_setB=frames_setB,
                surrogate_by_building=surrogate_by_building,
                num_scaler_ehr=num_scaler_ehr,
                ohe_ehr=ohe_ehr,
            )
            detail_df["seed"] = seed
            portfolio["seed"] = seed
            all_detail.append(detail_df)
            all_portfolio.append(portfolio)
            print(f"     portfolio WAPE={portfolio['wape_pct']:.3f}% | RMSE={portfolio['rmse']:.3f}")

    detail_all_df = pd.concat(all_detail, ignore_index=True)
    portfolio_all_df = pd.DataFrame(all_portfolio).sort_values(["seed", "variant"]).reset_index(drop=True)

    print("\nPortfolio results:")
    print(portfolio_all_df.to_string(index=False))



=== Running seed 42 ===
  -> variant: setA_base


ImportError: `keras.optimizers.legacy` is not supported in Keras 3. When using `tf.keras`, to continue using a `tf.keras.optimizers.legacy` optimizer, you can install the `tf_keras` package (Keras 2) and set the environment variable `TF_USE_LEGACY_KERAS=True` to configure TensorFlow to use `tf_keras` when accessing `tf.keras`.

In [ ]:
# Section 6 — Summary tables and EHR relative improvement

if "portfolio_all_df" not in globals() or portfolio_all_df.empty:
    print("No results yet. Run Section 5 first.")
else:
    portfolio_mean_df = (
        portfolio_all_df.groupby("variant", as_index=False)
        .agg(
            wape_mean=("wape_pct", "mean"),
            wape_std=("wape_pct", "std"),
            rmse_mean=("rmse", "mean"),
            r2_mean=("r2", "mean"),
            n_runs=("seed", "count"),
            n_train_seq=("n_train_seq", "mean"),
            n_val_seq=("n_val_seq", "mean"),
            n_test_seq=("n_test_seq", "mean"),
        )
        .sort_values("variant")
        .reset_index(drop=True)
    )

    print("Mean portfolio metrics by variant:")
    print(portfolio_mean_df.to_string(index=False))

    # EHR relative gain on portfolio WAPE
    wape_by_variant = portfolio_mean_df.set_index("variant")["wape_mean"].to_dict()

    ehr_relative_df = pd.DataFrame([
        {
            "comparison": "base",
            "setA_wape": wape_by_variant.get("setA_base", np.nan),
            "setB_wape": wape_by_variant.get("setB_base", np.nan),
            "ehr_relative_gain_pp": wape_by_variant.get("setA_base", np.nan) - wape_by_variant.get("setB_base", np.nan),
        },
        {
            "comparison": "surrogate",
            "setA_wape": wape_by_variant.get("setA_sur", np.nan),
            "setB_wape": wape_by_variant.get("setB_sur", np.nan),
            "ehr_relative_gain_pp": wape_by_variant.get("setA_sur", np.nan) - wape_by_variant.get("setB_sur", np.nan),
        },
    ])

    print("\nEHR relative gain summary (positive is better for EHR):")
    print(ehr_relative_df.to_string(index=False))

    # Building-level WAPE gains
    building_wape = (
        detail_all_df.groupby(["building", "variant"], as_index=False)["wape_pct"].mean()
        .pivot(index="building", columns="variant", values="wape_pct")
        .reset_index()
    )
    building_wape["ehr_gain_base_pp"] = building_wape["setA_base"] - building_wape["setB_base"]
    building_wape["ehr_gain_surrogate_pp"] = building_wape["setA_sur"] - building_wape["setB_sur"]

    print("\nBuilding-level EHR gains (pp):")
    print(building_wape[["building", "ehr_gain_base_pp", "ehr_gain_surrogate_pp"]].to_string(index=False))

    # Optional paired tests per seed (requires overlapping building rows)
    paired_stats_rows = []
    for seed in sorted(detail_all_df["seed"].unique()):
        tmp = detail_all_df[detail_all_df["seed"] == seed]
        piv = tmp.pivot(index="building", columns="variant", values="wape_pct")

        if {"setA_base", "setB_base"}.issubset(piv.columns):
            stat, p = wilcoxon(piv["setA_base"], piv["setB_base"], alternative="two-sided")
            paired_stats_rows.append({"seed": seed, "comparison": "base(setA vs setB)", "pvalue_two_sided": float(p)})

        if {"setA_sur", "setB_sur"}.issubset(piv.columns):
            stat, p = wilcoxon(piv["setA_sur"], piv["setB_sur"], alternative="two-sided")
            paired_stats_rows.append({"seed": seed, "comparison": "surrogate(setA vs setB)", "pvalue_two_sided": float(p)})

    paired_stats_df = pd.DataFrame(paired_stats_rows)
    if not paired_stats_df.empty:
        print("\nPaired Wilcoxon tests by seed:")
        print(paired_stats_df.to_string(index=False))


In [ ]:
# Section 7 — Plots: portfolio and building-level gains

if "portfolio_mean_df" not in globals() or portfolio_mean_df.empty:
    print("No summary tables yet. Run Section 6 first.")
else:
    # 7a. Portfolio WAPE by variant
    plt.figure(figsize=(8, 4))
    tmp = portfolio_mean_df.sort_values("variant")
    plt.bar(tmp["variant"], tmp["wape_mean"], yerr=tmp["wape_std"].fillna(0.0), capsize=4)
    plt.ylabel("WAPE [%]")
    plt.title("Portfolio WAPE by variant (mean across seeds)")
    plt.tight_layout()
    plt.show()

    # 7b. Building-level EHR gain comparison
    if "building_wape" in globals() and not building_wape.empty:
        plot_df = building_wape[["building", "ehr_gain_base_pp", "ehr_gain_surrogate_pp"]].melt(
            id_vars="building",
            var_name="metric",
            value_name="gain_pp",
        )
        plt.figure(figsize=(10, 5))
        sns.barplot(data=plot_df, x="building", y="gain_pp", hue="metric")
        plt.axhline(0.0, color="black", linewidth=1)
        plt.ylabel("EHR relative gain [pp] (positive = EHR better)")
        plt.title("Building-level EHR gain: base vs surrogate-enhanced path")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

    # 7c. Variant heatmap (building vs WAPE)
    if "detail_all_df" in globals() and not detail_all_df.empty:
        hm = detail_all_df.groupby(["building", "variant"], as_index=False)["wape_pct"].mean()
        hm_p = hm.pivot(index="building", columns="variant", values="wape_pct")
        plt.figure(figsize=(8, 5))
        sns.heatmap(hm_p, annot=True, fmt=".2f", cmap="YlOrRd")
        plt.title("Mean WAPE by building and variant")
        plt.tight_layout()
        plt.show()


## Section 8 — Interpretation Template

Use this structure when reporting outcomes:

1. **Surrogate effect**: Compare `setA_base` vs `setA_sur`.
2. **EHR effect without surrogate**: Compare `setA_base` vs `setB_base`.
3. **EHR effect with surrogate**: Compare `setA_sur` vs `setB_sur`.
4. **Key decision**:
   - If `ehr_relative_gain_pp` becomes less negative / positive in surrogate path,
     then stronger inertia-state context made EHR more useful.
   - If still negative, EHR likely remains more useful for XAI/context than for direct accuracy.

Important boundary:
- Surrogate indoor state is assumption-based latent proxy, not measured indoor temperature.


In [ ]:
# Section 9 — Export artifacts

if "detail_all_df" not in globals() or detail_all_df.empty:
    print("No experiment outputs to export yet.")
else:
    tag = f"lstm_surrogate_ehr_path_{SURROGATE_PRESET}_lb{CFG.lookback}_ep{CFG.epochs}_seeds{len(SEEDS)}"

    detail_path = RESULTS_DIR / f"{tag}_per_building.csv"
    portfolio_path = RESULTS_DIR / f"{tag}_portfolio_runs.csv"
    mean_path = RESULTS_DIR / f"{tag}_portfolio_mean.csv"
    ehr_gain_path = RESULTS_DIR / f"{tag}_ehr_gain_summary.csv"
    building_gain_path = RESULTS_DIR / f"{tag}_building_ehr_gain.csv"
    paired_path = RESULTS_DIR / f"{tag}_paired_wilcoxon.csv"

    detail_all_df.to_csv(detail_path, index=False)
    portfolio_all_df.to_csv(portfolio_path, index=False)

    if "portfolio_mean_df" in globals():
        portfolio_mean_df.to_csv(mean_path, index=False)
    if "ehr_relative_df" in globals():
        ehr_relative_df.to_csv(ehr_gain_path, index=False)
    if "building_wape" in globals():
        building_wape.to_csv(building_gain_path, index=False)
    if "paired_stats_df" in globals() and not paired_stats_df.empty:
        paired_stats_df.to_csv(paired_path, index=False)

    print("Saved outputs:")
    print(detail_path)
    print(portfolio_path)
    print(mean_path)
    print(ehr_gain_path)
    print(building_gain_path)
    if "paired_stats_df" in globals() and not paired_stats_df.empty:
        print(paired_path)
